##1. Create Widget

In [0]:
dbutils.widgets.text("source_system", "")

source_system = dbutils.widgets.get("source_system")

print(f"Source System Passed: {source_system}")

if not source_system.strip():
    raise ValueError("source_system widget cannot be empty")

## 2. Read Metadata Tables

In [0]:
tables_df = spark.table("banking.metadata.tables")

filtered_df = (
    tables_df
    .filter(f"active_flag = true AND lower(source_system) = lower('{source_system}')")
    .orderBy("load_order")
)

display(filtered_df)

## 3. Convert to List of Dictionaries
Convert Spark DataFrame rows into Python dictionaries to make the data easier to use in Python logic (loops, conditions, and dynamic ETL processing).

In [0]:
from datetime import datetime

rows = filtered_df.collect()

tables_list = [row.asDict() for row in rows]

# Convert datetime fields to string for JSON compatibility
for row in tables_list:
    if "created_at" in row and row["created_at"] is not None:
        row["created_at"] = str(row["created_at"])

print("Tables to Process (Full Metadata):")
print(tables_list)

## 4. Set Databricks Task Value
Stores metadata in the Databricks job context to share it between different tasks in the workflow.

In [0]:
dbutils.jobs.taskValues.set(
    key="tables_metadata",
    value=tables_list
)

print("Task value 'tables_metadata' has been set.")